# Module 3: Code Gen Agents & Reflexion

**Goal:** Build an autonomous agent that can write, execute, and fix its own code (Reflexion).

## 1. Setup
We need a safe way to execute code. For this notebook, we'll use a restricted `exec` environment, but in production, use E2B or Docker.

In [1]:
import io
import sys
import traceback
import contextlib
from typing import Optional, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
MODEL = "gpt-4o-mini"

print("Setup complete.")

Setup complete.


## 2. The Code Executor Tool
This tool takes Python code as a string, runs it, and captures `stdout` and `stderr`.

In [2]:
class CodeExecutor:
    def __init__(self, unsafe: bool = False):
        self.unsafe = unsafe
        self.globals = {}

    def run(self, code: str) -> str:
        """Executes Python code and returns the output or error."""
        buffer = io.StringIO()
        
        try:
            # Capture stdout
            with contextlib.redirect_stdout(buffer):
                exec(code, self.globals)
            return buffer.getvalue()
        except Exception:
            return traceback.format_exc()

# Test it
executor = CodeExecutor()
code = """
def factorial(n):
    return 1 if n == 0 else n * factorial(n-1)

print(factorial(5))
"""
print(executor.run(code))

120



## 3. The Code Agent (Generator)
We define the system prompt to encourage the agent to write clean, executable Python.

In [3]:
SYSTEM_PROMPT = """
You are an expert Python programmer. 
Your task is to write Python code to solve the user's problem. 
Wrap your code in markdown code blocks, e.g., ```python ... ```.
Do not assume any external libraries are installed aside from standard library.
Always include a print statement to show the result.
"""

def generate_code(problem: str, history: list = None) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": problem})
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    return response.choices[0].message.content

def extract_code(text: str) -> str:
    """Extracts code from markdown blocks."""
    if "```python" in text:
        return text.split("```python")[1].split("```")[0].strip()
    elif "```" in text:
        return text.split("```")[1].split("```")[0].strip()
    return text

# Test Generation
problem = "Calculate the 10th Fibonacci number."
raw_response = generate_code(problem)
clean_code = extract_code(raw_response)
print(f"Generated Code:\n{clean_code}")

Generated Code:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

tenth_fibonacci = fibonacci(10)
print("The 10th Fibonacci number is:", tenth_fibonacci)


## 4. The Reflexion Loop
If the code fails, feed the error back to the agent and ask it to fix it.

In [4]:
def solve_problem(problem: str, max_retries: int = 3):
    history = []
    
    for attempt in range(max_retries):
        print(f"--- Attempt {attempt + 1} ---")
        
        # 1. Generate
        if attempt == 0:
            response = generate_code(problem)
        else:
            # Feedback Loop
            response = generate_code("Fix the code based on the error above.", history)
            
        code = extract_code(response)
        print("Running code...")
        
        # 2. Execute
        output = executor.run(code)
        
        if "Traceback" in output:
            print(f"❌ Error:\n{output}")
            history.append({"role": "assistant", "content": response})
            history.append({"role": "user", "content": f"The code failed with this error:\n{output}"})
        else:
            print(f"✅ Success:\n{output}")
            return output
            
    print("❌ Failed to solve problem after max retries.")
    return None

# Test with a tricky prompt (deliberately broken logic often helps testing, but let's try a standard one)
solve_problem("Create a list of 5 random numbers and sort them in descending order.")

--- Attempt 1 ---
Running code...
✅ Success:
Random numbers: [62, 38, 37, 52, 62]
Sorted numbers in descending order: [62, 62, 52, 38, 37]



'Random numbers: [62, 38, 37, 52, 62]\nSorted numbers in descending order: [62, 62, 52, 38, 37]\n'

## 5. Benchmarking (Pass@k Concept)
We can evaluate our agent by running it on multiple problems and counting the success rate.

In [5]:
problems = [
    "Print 'Hello World' reversed.",
    "Calculate the sum of the first 50 prime numbers.",
    "Find the common elements between two lists: [1, 2, 3] and [2, 3, 4]."
]

for p in problems:
    print(f"\nProblem: {p}")
    solve_problem(p, max_retries=2)


Problem: Print 'Hello World' reversed.
--- Attempt 1 ---
Running code...
✅ Success:
dlroW olleH


Problem: Calculate the sum of the first 50 prime numbers.
--- Attempt 1 ---
Running code...
✅ Success:
The sum of the first 50 prime numbers is: 5117


Problem: Find the common elements between two lists: [1, 2, 3] and [2, 3, 4].
--- Attempt 1 ---
Running code...
✅ Success:
Common elements: [2, 3]

